In [ ]:
import os
import scipy.io
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import ResNet50V2, EfficientNetB0, MobileNetV3Large
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)
print("GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# --- CONFIGURATION ---
BASE_DIR = "."  # Change this if your data is in a subfolder
IMAGES_DIR = os.path.join(BASE_DIR, "Images")
TRAIN_MAT = os.path.join(BASE_DIR, "train_list.mat")
TEST_MAT = os.path.join(BASE_DIR, "test_list.mat")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# --- PARSE .MAT FILES ---
def load_split_from_mat(mat_path):
    mat_data = scipy.io.loadmat(mat_path)
    # Extract file paths and labels (labels are 1-indexed in MATLAB, we need 0-indexed)
    file_paths = [os.path.join(IMAGES_DIR, str(f[0][0])) for f in mat_data['file_list']]
    labels = [int(l[0]) - 1 for l in mat_data['labels']]
    return file_paths, labels

train_paths, train_labels = load_split_from_mat(TRAIN_MAT)
test_paths, test_labels = load_split_from_mat(TEST_MAT)
NUM_CLASSES = len(np.unique(train_labels))
print(f"Loaded {len(train_paths)} Train images and {len(test_paths)} Test images across {NUM_CLASSES} classes.")

# --- CREATE TF.DATA PIPELINE ---
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    # We do NOT normalize to [0,1] here because Keras pre-trained models handle their own scaling
    return img, label

# Data Augmentation layer (runs on GPU)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(len(train_paths)).map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_model(base_model_name):
    # Select the pre-trained base model
    if base_model_name == "ResNet50":
        base_model = ResNet50V2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
        preprocess_input = tf.keras.applications.resnet_v2.preprocess_input
    elif base_model_name == "EfficientNetB0":
        base_model = EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
        preprocess_input = tf.keras.applications.efficientnet.preprocess_input
    elif base_model_name == "MobileNetV3":
        base_model = MobileNetV3Large(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
        preprocess_input = tf.keras.applications.mobilenet_v3.preprocess_input

    # Freeze the base model so we don't destroy the pre-trained weights
    base_model.trainable = False

    # Build the custom head
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = preprocess_input(inputs) # Automatically handles scaling specific to the model
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x) # Prevents overfitting
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs, name=base_model_name)
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
# List of models to train
model_names = ["MobileNetV3", "ResNet50", "EfficientNetB0"]
training_histories = {}

for name in model_names:
    print(f"\n{'='*50}\nTraining {name}\n{'='*50}")
    model = build_model(name)
    
    # 1. Early Stopping: Stop if validation accuracy doesn't improve for 5 epochs
    early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
    
    # 2. Model Checkpoint: Save the best version of the model to disk
    checkpoint = callbacks.ModelCheckpoint(filepath=f"best_{name}.keras", monitor='val_accuracy', save_best_only=True)
    
    # 3. Learning Rate Scheduler: Reduce LR if the model stops learning
    lr_scheduler = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
    
    # Train the model (Set to 30 epochs, but Early Stopping will likely stop it around epoch 10-15)
    history = model.fit(
        train_ds,
        validation_data=test_ds,
        epochs=30, 
        callbacks=[early_stop, checkpoint, lr_scheduler]
    )
    
    training_histories[name] = history.history

In [ ]:
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Load the best EfficientNet model we saved
best_model = tf.keras.models.load_model("best_EfficientNetB0.keras")

# Pick a random test image
sample_img_path = test_paths[10] 
img = tf.keras.preprocessing.image.load_img(sample_img_path, target_size=(224, 224))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array_exp = np.expand_dims(img_array, axis=0)

# The name of the last convolutional layer in EfficientNetB0
last_conv_layer = "top_activation"

heatmap = make_gradcam_heatmap(img_array_exp, best_model.layers[2], last_conv_layer)

# Overlay heatmap on original image
heatmap = cv2.resize(heatmap, (img_array.shape[1], img_array.shape[0]))
heatmap = np.uint8(255 * heatmap)
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
superimposed_img = heatmap * 0.4 + img_array
superimposed_img = tf.keras.preprocessing.image.array_to_img(superimposed_img)

# Plotting
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img)
ax[0].set_title("Original Image")
ax[0].axis("off")
ax[1].imshow(superimposed_img)
ax[1].set_title("Grad-CAM Heatmap")
ax[1].axis("off")
plt.show()